<a href="https://colab.research.google.com/github/lelongc/rac/blob/main/ace_step_1-5-customi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import sys

# 1. Clone repo nếu chưa có
if not os.path.exists("Ace-Step-v1.5"):
    !git clone https://huggingface.co/spaces/ACE-Step/Ace-Step-v1.5
os.chdir("/content/Ace-Step-v1.5")

# 2. Vá lỗi phiên bản Torch và bật tính năng share public
!sed -i 's/torch>=2.9.1/torch/g' requirements.txt
!sed -i 's/share=False/share=True/g' app.py

# 3. Cài đặt nano-vllm THỦ CÔNG (Đây là chìa khóa chống crash RAM)
print("📦 Đang cài đặt lõi tăng tốc nano-vllm...")
os.chdir("/content/Ace-Step-v1.5/acestep/third_parts/nano-vllm")
!pip install -e .
os.chdir("/content/Ace-Step-v1.5")

print("✅ Bước 1 xong. Đã cài lõi tăng tốc.")

Cloning into 'Ace-Step-v1.5'...
remote: Enumerating objects: 1298, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 1298 (delta 0), reused 0 (delta 0), pack-reused 1295 (from 1)
Receiving objects: 100% (1298/1298), 1.57 MiB | 6.13 MiB/s, done.
Resolving deltas: 100% (789/789), done.
Filtering content: 100% (7/7), 2.04 MiB | 1.03 MiB/s, done.
📦 Đang cài đặt lõi tăng tốc nano-vllm...
Obtaining file:///content/Ace-Step-v1.5/acestep/third_parts/nano-vllm
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 76.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Building editable for nano-vllm (pyproject.toml) ... done
  Created wheel for nano-vllm: filename=nano_vllm-0.2.0-0.editable-py3-none-any.whl size=5037 s

In [ ]:
import os
os.chdir("/content/Ace-Step-v1.5")

print("Installing Python dependencies...")
!pip install -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu121
!apt-get install -y ffmpeg

print("✅ Bước 2 xong.")

Installing Python dependencies...
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu121, https://download.pytorch.org/whl/cu128
Ignoring torch: markers 'sys_platform == "win32"' don't match your environment
Ignoring torchaudio: markers 'sys_platform == "win32"' don't match your environment
Ignoring torchvision: markers 'sys_platform == "win32"' don't match your environment
Ignoring triton-windows: markers 'sys_platform == "win32"' don't match your environment
Ignoring flash-attn: markers 'sys_platform == "win32" and python_version == "3.11" and platform_machine == "AMD64"' don't match your environment
Ignoring flash-attn: markers 'sys_platform == "linux" and python_version == "3.11"' don't match your environment
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.1/139.1 MB 126.2 MB/s eta 0:00:00
INFO: p

In [ ]:
import os
import torch
import gc

os.chdir("/content/Ace-Step-v1.5")

# --- CẤU HÌNH SIÊU TIẾT KIỆM (ANTI-CRASH) ---
os.environ["SERVICE_MODE_DIT_MODEL_2"] = ""        # Tắt model phụ
os.environ["SERVICE_MODE_BACKEND"] = "vllm"        # Ép dùng vllm (nhẹ hơn)
os.environ["MAX_MODEL_LEN"] = "512"               # Giới hạn độ dài nhạc để cứu RAM
os.environ["VLLM_GPU_MEMORY_UTILIZATION"] = "0.4" # Chỉ lấy 40% GPU làm cache, nhường RAM cho hệ thống
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Dọn rác RAM hệ thống trước khi chạy
gc.collect()
torch.cuda.empty_cache()

# 1. Tải model (Nếu chưa có)
if not os.path.exists("/content/Ace-Step-v1.5/data/checkpoints"):
    print("⏳ Đang tải model 10GB (Chỉ tải 1 lần duy nhất)...")
    download_code = """
import os, sys, torch
from acestep.handler import AceStepHandler
current_dir = os.getcwd()
sys.path.insert(0, os.path.join(current_dir, "acestep", "third_parts", "nano-vllm"))
handler = AceStepHandler(persistent_storage_path=os.path.join(current_dir, "data"))
handler.initialize_service(current_dir, "acestep-v15-turbo", device='cpu', offload_to_cpu=True)
"""
    with open("download_models.py", "w") as f:
        f.write(download_code)
    !python download_models.py

# 2. Khởi chạy app
print("="*60)
print("🚀 ĐANG KHỞI CHẠY ACE-STEP TRÊN CHẾ ĐỘ TIẾT KIỆM RAM...")
print("🔗 Hãy đợi link 'gradio.live' hiện ra bên dưới.")
print("="*60)

!python app.py